# Pipeline to process big satellite images

This notebook allows you to generates part of the dataset for:

* Jacksonville
* Omaha
* UCSD

It does the following tasks:

1. Compute the AOIs based on the satellite images footprints and keeps areas where at least 2 images are available.
2. Creates 500 m x 500 m tiles of the AOIs while removing tiles that are not interested (mostly water)
3. Creates all the crops for a tile and the metadata json with rpcs.
3. Downloads LIDAR point clouds from USGS and builds the DSM (using pdal and gdal)

In [ ]:
# IPython magics
%load_ext autoreload
%autoreload 2


# Standard library imports
import os
import json
from itertools import combinations

# Third-party library imports
import numpy as np
import matplotlib.pyplot as plt
import geojson
import ipyleaflet
import requests
import srtm4
import geopandas as gpd
import rasterio
import pdal
from shapely.geometry import shape, box
from shapely.ops import unary_union
import rpcm

# Project-specific imports (custom modules)
import pdal_utils  # PDAL tools
import utils       # IO tools
import vistools    # display tools

# Configure numpy print options
np.set_printoptions(linewidth=150)

# Visualize all images in a map and extract AOIs

AOIs are zones where at least 2 images are available

In [ ]:
default_crs = "epsg:4326"

jax_files = utils.find('PRIVATE_URL_TO_CORE3D_DATA/Jacksonville/WV3/PAN/', '.NTF.tif')
oma_files = utils.find('PRIVATE_URL_TO_CORE3D_DATA/Omaha/WV3/PAN/', '.NTF')
ucsd_files = utils.find('PRIVATE_URL_TO_CORE3D_DATA/UCSD/WV3/PAN/', '.NTF')

files = {'OMA': oma_files, 'UCSD': ucsd_files, 'JAX': jax_files, }
files = {k: sorted(v, key=lambda x: utils.acquisition_date(x)) for k, v in files.items()}

waterbody_files = {'JAX': 'data/water_masks/JAX/w082n30n.shp', 'OMA': 'data/water_masks/OMA/w096n41n.shp', 'UCSD': 'data/water_masks/UCSD/w118n32n.shp'}

utm_zones = {'OMA': 14, 'UCSD': 11, 'JAX': 17}

In [ ]:
for k, v in files.items():
    print(k, len(v))

In [ ]:
def create_footprint_union(city_files):
    individual_footprints = []
    
    for f in city_files:
        footprint = utils.lon_lat_image_footprint(f)        
        polygon = shape(footprint)
        individual_footprints.append(polygon)

    union = unary_union(individual_footprints)

    return union

def create_footprint_intersection(city_files):
    individual_footprints = []

    for f in city_files:
        footprint = utils.lon_lat_image_footprint(f)
        polygon = shape(footprint)
        individual_footprints.append(polygon)
    intersection = individual_footprints[0]

    for footprint in individual_footprints[1:]:
        intersection = intersection.intersection(footprint)    

    return intersection


def create_footprint_pairwise_intersections(city_files):
    individual_footprints = []
    
    for f in city_files:
        footprint = utils.lon_lat_image_footprint(f)        
        polygon = shape(footprint)
        individual_footprints.append(polygon)

    pairwise_intersection = [a.intersection(b) for a, b in combinations(individual_footprints, 2)]
    pairwise_intersection = unary_union(pairwise_intersection)
    return pairwise_intersection

m = vistools.clickablemap()

aois = {}


for city, city_files in files.items():
    for f in city_files:
        footprint = utils.lon_lat_image_footprint(f)
        m.add_layer(ipyleaflet.GeoJSON(data=footprint, style={'color': 'blue', 'weight': 1, 'fillOpacity': 0, 'fillColor': 'transparent'}))

    union = create_footprint_union(city_files)
    union_feature = geojson.Feature(geometry=union.__geo_interface__)
    intersection = create_footprint_intersection(city_files)
    intersection_feature = geojson.Feature(geometry=intersection.__geo_interface__)
    pairwise_intersection = create_footprint_pairwise_intersections(city_files)
    pairwise_intersection_feature = geojson.Feature(geometry=pairwise_intersection.__geo_interface__)
    # m.add_layer(ipyleaflet.GeoJSON(data=union_feature, style={'color': 'red', 'weight': 1, 'opacity': 0.05}))
    # m.add_layer(ipyleaflet.GeoJSON(data=intersection_feature, style={'color': 'green', 'weight': 1, 'opacity': 0.05}))
    m.add_layer(ipyleaflet.GeoJSON(data=pairwise_intersection_feature, style={'color': 'purple', 'weight': 1, 'opacity': 0.05}))
    aois[city] = pairwise_intersection

# Put the center in the center of the last footprint intersection
m.center = list(intersection.centroid.coords)[0][::-1]

display(m)

# Define all the tiles for a city based on the MBR

In [ ]:
def create_tiles(mbr_gdf, tile_size=500):
    # Ensure we're working in a projected CRS
    if mbr_gdf.crs.is_geographic:
        raise ValueError("MBR must be in a projected coordinate system")
    
    mbr = mbr_gdf.geometry.iloc[0]
    minx, miny, maxx, maxy = mbr.bounds
    
    tiles = []
    x_coords = np.arange(minx, maxx + tile_size, tile_size)
    y_coords = np.arange(miny, maxy + tile_size, tile_size)
    
    for x in x_coords[:-1]:
        for y in y_coords[:-1]:
            tile = box(x, y, x + tile_size, y + tile_size)
            tiles.append(tile)
    
    tiles_gdf = gpd.GeoDataFrame(geometry=tiles, crs=mbr_gdf.crs)
    return tiles_gdf

def process_and_convert(aoi_gdf, utm_crs, water_gdf, tile_size=500, water_threshold=.8):
    # Convert AOI to UTM
    aoi_utm = aoi_gdf.to_crs(utm_crs)

    # Convert water mask to UTM
    water_utm = water_gdf.to_crs(utm_crs)

    # Create MBR in UTM
    mbr_utm = aoi_utm.envelope

    # Create tiles in UTM
    tiles_utm = create_tiles(mbr_utm, tile_size)

    # Remove tiles that are mostly water
    # TODO: remove this part because the filter doesn't seem to be working as expected
    # There are some tiles removed that almost entirely contain land ...
    intersections = tiles_utm.overlay(water_utm, how='intersection')
    intersections['area'] = intersections.area
    water_tiles = intersections[intersections['area'] > water_threshold * tiles_utm.area.max()]
    non_water_tiles = tiles_utm.sjoin(water_tiles, how='left', predicate='intersects')
    non_water_tiles = non_water_tiles[non_water_tiles['index_right'].isna()]
    tiles_utm = non_water_tiles.drop(columns='index_right')

    # Convert everything back to the original CRS
    mbr_original = mbr_utm.to_crs(aoi_gdf.crs)
    tiles_original = tiles_utm.to_crs(aoi_gdf.crs)

    return mbr_original, tiles_original, tiles_utm

city = 'JAX'
utm_crs = f"EPSG:326{utm_zones[city]}"

# load water mask to exclude water only tiles
waterbody_file = waterbody_files[city]
waterbody_gdf = gpd.read_file(waterbody_file)
waterbody_gdf.crs = default_crs

aoi_gdf = gpd.GeoDataFrame(geometry=[aois[city]], crs=default_crs)
mbr, tiles, tiles_utm = process_and_convert(aoi_gdf, utm_crs, waterbody_gdf, tile_size=500)


# Create map
m = vistools.clickablemap()
m.add_layer(ipyleaflet.GeoJSON(data=waterbody_gdf.__geo_interface__, style={'color': 'blue', 'weight': 1, 'fillOpacity': 0.5, 'fillColor': 'blue'}))
m.add_layer(ipyleaflet.GeoJSON(data=mbr.__geo_interface__, style={'color': 'green', 'weight': 5, 'fillOpacity': 1, 'fillColor': 'transparent'}))
tiles_geojson = tiles.__geo_interface__
m.add_layer(ipyleaflet.GeoJSON(
    data=tiles_geojson,
    style={
        'color': 'red',
        'weight': 1,
        'fillOpacity': 0.1,
        'fillColor': 'transparent'
    }
))
m.center = list(aoi_gdf.geometry.iloc[0].centroid.coords)[0][::-1]
display(m)

print(f"A total of {len(tiles)} tiles were created.")

# Extract all crops for a tile

In [ ]:
# Select a tile for processing
n_tile = 370
tile = tiles.iloc[n_tile].geometry
tile_utm = tiles_utm.iloc[n_tile].geometry

In [ ]:
# Plot the tile on top of the map
m = vistools.clickablemap()

# Convert tile to GeoJSON format
tile_geojson = geojson.Feature(geometry=tile.__geo_interface__)

# Add tile to map
m.add_layer(ipyleaflet.GeoJSON(data=tile_geojson, style={'color': 'red', 'weight': 1, 'opacity': 0.5}))

# Define center using the centroid of the tile
center = list(tile.centroid.coords)[0]
m.center = center[::-1]

m.zoom = 15
display(m)

In [ ]:
def tile_in_image(tile, image):
    corners = [(lon, lat) for lon, lat in tile.__geo_interface__['coordinates'][0]]
    lons = [lon for lon, _ in corners]
    lats = [lat for _, lat in corners]
    z = srtm4.srtm4(tile.centroid.x, tile.centroid.y)
    xs_img, ys_img = rpcm.projection(image, lons, lats, z)
    metadata = utils.readGTIFFmeta(image)[0]
    # If all xs_imgs are in the image, the tile is in the image
    return all([0 <= x < metadata['width'] for x in xs_img]) and all([0 <= y < metadata['height'] for y in ys_img])

def crop_tile_in_image(tile, image):
    z = srtm4.srtm4(tile.centroid.x, tile.centroid.y)
    return utils.crop_aoi(image, tile.__geo_interface__, z)

def crop_geotiff_lonlat_aoi(geotiff_path, output_path, tile):
    z = srtm4.srtm4(tile.centroid.x, tile.centroid.y)
    with rasterio.open(geotiff_path, 'r') as src:
        profile = src.profile
        tags = src.tags()
    crop, x, y = rpcm.utils.crop_aoi(geotiff_path, tile.__geo_interface__, z)
    rpc = rpcm.rpc_from_geotiff(geotiff_path)
    rpc.row_offset -= y
    rpc.col_offset -= x
    not_pan = len(crop.shape) > 2
    if not_pan:
        profile["height"] = crop.shape[1]
        profile["width"] = crop.shape[2]
    else:
        profile["height"] = crop.shape[0]
        profile["width"] = crop.shape[1]
        profile["count"] = 1
    with rasterio.open(output_path, 'w', **profile) as dst:
        if not_pan:
            dst.write(crop)
        else:
            dst.write(crop, 1)
        dst.update_tags(**tags)
        dst.update_tags(ns='RPC', **rpc.to_geotiff_dict())

In [ ]:
out_dir = "/mnt/adisk/datasets/shadow_dataset"
os.makedirs(out_dir, exist_ok=True)

crops_dir = os.path.join(out_dir, "crops", f"{city.upper()}_{n_tile}")
os.makedirs(crops_dir, exist_ok=True)
root_dir = os.path.join(out_dir, "root_dir", f"{city.upper()}_{n_tile}")
os.makedirs(root_dir, exist_ok=True)

In [ ]:
files_containing_crop = []
index_containing_crop = []
crops = []
metas = []

# First pass, check in how many images the tile is
for img_index, img_file in enumerate(jax_files):
    if tile_in_image(tile, img_file):
        files_containing_crop.append(img_file)
        index_containing_crop.append(img_index)

# Second pass, if enough images, extract the crops
if len(files_containing_crop) > len(jax_files) // 2:
    for i, img_file in enumerate(files_containing_crop):
        crop_name = f"{city.upper()}_{n_tile}_{index_containing_crop[i]}.tif"

        # Crop the tile in the image and save it (along with a modified RPC)
        crop_geotiff_lonlat_aoi(img_file, os.path.join(crops_dir, crop_name), tile)

        crops.append(
            utils.readGTIFF(os.path.join(crops_dir, crop_name)).squeeze()
        )

        # Generate metadata (everything but the DSM related stuff, that will be done later)
        meta = utils.generate_json_metadata(os.path.join(crops_dir, crop_name))
        metas.append(meta)
        meta_name = os.path.join(root_dir, meta["img"].replace(".tif", ".json"))
        json.dump(meta, open(meta_name, "w"))

In [ ]:
# Plot all the crops
print(f'Found {len(crops)} crops')
for crop, fi in zip(crops, index_containing_crop):
    plt.figure()
    plt.imshow(np.sqrt(crop).astype(np.uint16), cmap='gray')
    plt.title(f"{city.upper()}_{n_tile}_{fi}.tif")

# Build a DSM and point cloud for a tile

In [ ]:
def get_3dep_data(url='https://raw.githubusercontent.com/hobuinc/usgs-lidar/master/boundaries/resources.geojson'):
    """
    Fetch 3DEP data and process it into a GeoDataFrame using EPSG:4326
    """
    geojsons_3DEP = requests.get(url).json()
    
    # Create GeoDataFrame with EPSG:4326 CRS
    df_3DEP = gpd.GeoDataFrame.from_features(geojsons_3DEP['features'], crs='EPSG:4326')
    
    return df_3DEP

def find_intersecting_polys(df_3DEP, target_geom):
    """
    Find polygons in df_3DEP that intersect with the target geometry.
    
    :param df_3DEP: GeoDataFrame containing 3DEP data
    :param target_geom: Target geometry to check for intersections (assumed to be in EPSG:4326)
    :return: GeoDataFrame of intersecting polygons
    """
    # Validation step
    if df_3DEP.crs != 'EPSG:4326':
        raise ValueError("Input GeoDataFrame must use EPSG:4326 coordinate system")
    
    if not isinstance(target_geom, (gpd.GeoSeries, gpd.GeoDataFrame)):
        try:
            target_geom = gpd.GeoSeries([shape(target_geom)], crs='EPSG:4326')
        except:
            raise ValueError("target_geom must be a valid GeoJSON-like dict, GeoSeries, or GeoDataFrame")
    elif target_geom.crs != 'EPSG:4326':
        raise ValueError("target_geom must use EPSG:4326 coordinate system")
    
    # Ensure target_geom is a GeoSeries
    if isinstance(target_geom, gpd.GeoDataFrame):
        target_geom = target_geom.geometry
    
    # Find intersecting polygons
    intersecting = df_3DEP[df_3DEP.intersects(target_geom.iloc[0])]
    
    return intersecting

In [ ]:
df_3DEP = get_3dep_data()

intersecting_polys = find_intersecting_polys(df_3DEP, tile.__geo_interface__)
print(f"Number of intersecting polygons for aoi: {len(intersecting_polys)}")
print(intersecting_polys.head())
print("##############################################")

In [ ]:
# Find AOI center for plotting purposes
centroid = tile.centroid.coords[0]

# Create ipyleaflet map
m = vistools.clickablemap()
m.center = centroid[::-1]
# Add intersecting 3DEP polygons to the map
wlayer_3DEP_list = []
usgs_3dep_datasets = []
number_pts_est = []

for _, poly in intersecting_polys.iterrows():
    wlayer_3DEP = ipyleaflet.WKTLayer(
        wkt_string=poly['geometry'].wkt,
        style={"color": "green"}
    )
    m.add_layer(wlayer_3DEP)
    wlayer_3DEP_list.append(wlayer_3DEP)
    usgs_3dep_datasets.append(poly['name'])

    # Estimate total points using ratio of area and point count
    transformed_tile_geom = gpd.GeoSeries([tile], crs='EPSG:4326').iloc[0]
    total_points = (int((transformed_tile_geom.area / poly['geometry'].area) * poly['count']))
    number_pts_est.append(total_points)

# Add tile boundary to the map
m.add_layer(ipyleaflet.GeoJSON(data=tile.__geo_interface__))

# Sum the estimates of the number of points from each 3DEP dataset within the tile
num_pts_est = sum(number_pts_est)

m.zoom = 100

display(m)
print()
print(f'Your tile at full resolution will include approximately {num_pts_est} points')
for number_pts, d in zip(number_pts_est, usgs_3dep_datasets):
    print(f"Estimated number of points in dataset {d}: {number_pts}")
print()


In [ ]:
dsm_dir = os.path.join(out_dir, "dsm", f"{city.upper()}_{n_tile}")
os.makedirs(dsm_dir, exist_ok=True)

# Transform tile to EPSG:3857 to make the pipeline
transformed_tile_geom = gpd.GeoSeries([tile], crs='EPSG:4326').to_crs('EPSG:3857').iloc[0]

grid_resolution = 0.5

dsm_2007_file = os.path.join(dsm_dir, f'{city.upper()}_{n_tile}_resolution_{str(grid_resolution).replace(".", "")}_2007')
dsm_2017_file = os.path.join(dsm_dir, f'{city.upper()}_{n_tile}_resolution_{str(grid_resolution).replace(".", "")}_2017')


# Extract each point cloud separately
dsm_pipeline_2007 = pdal_utils.make_DEM_pipeline(
    extent_epsg3857=transformed_tile_geom,
    usgs_3dep_dataset_name=[usgs_3dep_datasets[0]],
    pc_resolution=0.1, # 100 points per square meter
    dem_resolution=grid_resolution, # 0.5 meter resolution in the grid
    filterNoise=True,
    reclassify=False,
    savePointCloud=True,
    outCRS=f"326{utm_zones[city]}",
    pc_outName=dsm_2007_file,
    pc_outType='laz',
    demType='dsm',
    gridMethod='idw',
    dem_outName=dsm_2007_file,
    dem_outExt='tif',
    driver="GTiff"
)

dsm_pipeline_2017 = pdal_utils.make_DEM_pipeline(
    extent_epsg3857=transformed_tile_geom,
    usgs_3dep_dataset_name=[usgs_3dep_datasets[1]],
    pc_resolution=0.1,
    dem_resolution=grid_resolution,
    filterNoise=True,
    reclassify=False,
    savePointCloud=True,
    outCRS=f"326{utm_zones[city]}",
    pc_outName=dsm_2017_file,
    pc_outType='laz',
    demType='dsm',
    gridMethod='idw',
    dem_outName=dsm_2017_file,
    dem_outExt='tif',
    driver="GTiff"
)

In [ ]:
dsm_pipeline = pdal.Pipeline(json.dumps(dsm_pipeline_2007))
dsm_pipeline.execute()

dsm_pipeline = pdal.Pipeline(json.dumps(dsm_pipeline_2017))
dsm_pipeline.execute()

In [ ]:
# Open both DSMs

with rasterio.open(dsm_2007_file + ".tif") as src:
    dsm_2007 = src.read(masked=True)
    dsm_meta_2007 = src.meta

with rasterio.open(dsm_2017_file + ".tif") as src:
    dsm_2017 = src.read(masked=True)
    dsm_meta_2017 = src.meta


# Plot them side by side
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(dsm_2007.T, cmap='gist_earth')
ax[0].set_title('2007 DSM')
ax[1].imshow(dsm_2017.T, cmap='gist_earth')
ax[1].set_title('2017 DSM')
# Include colorbars
fig.colorbar(ax[0].imshow(dsm_2007.T, cmap='gist_earth'), ax=ax[0])
fig.colorbar(ax[1].imshow(dsm_2017.T, cmap='gist_earth'), ax=ax[1])
plt.show()

In [ ]:
# Find min and max altitude in 2017 DSM, ignore masked values
min_altitude = np.min(dsm_2017[~dsm_2017.mask])
max_altitude = np.max(dsm_2017[~dsm_2017.mask])
print(f"Min altitude in 2017 DSM: {min_altitude}")
print(f"Max altitude in 2017 DSM: {max_altitude}")

In [ ]:
# Now yes, put the dsm info into the root dir jsons
for meta_json in os.listdir(root_dir):
    meta = json.load(open(os.path.join(root_dir, meta_json)))
    meta["min_alt"] = int(np.round(min_altitude -1))
    meta["max_alt"] = int(np.round(max_altitude + 1))
    json.dump(meta, open(os.path.join(root_dir, meta_json), "w"))